# 04 · Consulta y análisis básico de caudales

Este notebook consulta una estación hidrológica desde la API de FluvioTech y realiza:

- descarga de serie observada y simulada;
- limpieza básica de fechas;
- gráfico de hidrograma;
- resumen estadístico;
- métricas simples de desempeño.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

URL_CAUDALES = "https://fluviotech.com/hydrostations/api/series/"

In [ ]:
# Parámetros de consulta
# gauge_id identifica la estación hidrológica.
params = {
    "gauge_id": "PE_1113346",
    "start": "1981-01-01",
    "end": "2025-12-31"
}

response = requests.get(URL_CAUDALES, params=params)

print("Código de estado:", response.status_code)
print("URL consultada:", response.url)

response.raise_for_status()
result = response.json()

print("Estación:", result.get("gauge_name"))
print("Claves de respuesta:", result.keys())

In [ ]:
df = pd.DataFrame(result["data"])
df["date"] = pd.to_datetime(df["date"])

# Ordenamos por fecha por seguridad
df = df.sort_values("date").reset_index(drop=True)

print(df.shape)
df.head()

## 1. Hidrograma observado y simulado

In [ ]:
plt.figure(figsize=(11, 4))

if "flow_obs" in df.columns:
    plt.plot(df["date"], df["flow_obs"], label="Caudal observado")

if "flow_sim" in df.columns:
    plt.plot(df["date"], df["flow_sim"], label="Caudal simulado")

plt.ylabel("Caudal (m³/s)")
plt.xlabel("Fecha")
plt.title(result.get("gauge_name", "Estación hidrológica"))
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 2. Resumen estadístico

In [ ]:
columnas_caudal = [c for c in ["flow_obs", "flow_sim"] if c in df.columns]

df[columnas_caudal].describe()

## 3. Filtro por periodo

In [ ]:
# Ejemplo: seleccionar un periodo específico para observar con más detalle.
inicio = "2017-01-01"
fin = "2017-12-31"

df_periodo = df[(df["date"] >= inicio) & (df["date"] <= fin)]

plt.figure(figsize=(11, 4))

if "flow_obs" in df_periodo.columns:
    plt.plot(df_periodo["date"], df_periodo["flow_obs"], label="Caudal observado")

if "flow_sim" in df_periodo.columns:
    plt.plot(df_periodo["date"], df_periodo["flow_sim"], label="Caudal simulado")

plt.ylabel("Caudal (m³/s)")
plt.xlabel("Fecha")
plt.title(f"Hidrograma {inicio} a {fin}")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 4. Métricas simples de comparación

In [ ]:
def nash_sutcliffe(obs, sim):
    """Calcula NSE entre serie observada y simulada."""
    obs = pd.Series(obs).astype(float)
    sim = pd.Series(sim).astype(float)

    valid = obs.notna() & sim.notna()
    obs = obs[valid]
    sim = sim[valid]

    return 1 - ((obs - sim) ** 2).sum() / ((obs - obs.mean()) ** 2).sum()


def bias_percent(obs, sim):
    """Calcula sesgo porcentual: positivo = sobreestimación."""
    obs = pd.Series(obs).astype(float)
    sim = pd.Series(sim).astype(float)

    valid = obs.notna() & sim.notna()
    obs = obs[valid]
    sim = sim[valid]

    return 100 * (sim.sum() - obs.sum()) / obs.sum()

In [ ]:
if {"flow_obs", "flow_sim"}.issubset(df.columns):
    nse = nash_sutcliffe(df["flow_obs"], df["flow_sim"])
    pbias = bias_percent(df["flow_obs"], df["flow_sim"])

    print("NSE:", round(nse, 3))
    print("PBIAS (%):", round(pbias, 2))
else:
    print("No se encontraron ambas columnas: flow_obs y flow_sim.")

## Ejercicio

1. Cambia `gauge_id` por otra estación disponible.
2. Consulta solo un periodo de 5 años.
3. Grafica el hidrograma.
4. Calcula el caudal medio anual.

In [ ]:
# Caudal medio anual
if "flow_obs" in df.columns:
    df_anual = (
        df
        .assign(year=df["date"].dt.year)
        .groupby("year", as_index=False)["flow_obs"]
        .mean()
        .rename(columns={"flow_obs": "caudal_medio_anual_obs"})
    )

    df_anual.head()